# 01 · Recolección de titulares: La República y El Comercio

**Pregunta:** ¿Podemos juntar 200 titulares de cada diario, en cuatro categorías, para comparar y entrenar un clasificador?

Opción **C (comparada)** de la tarea: 200 de La República + 200 de El Comercio.
Este notebook se corre **una vez al día**: cada corrida agrega las notas nuevas a `datos/noticias.csv` sin repetir ninguna.

In [1]:
import time
from pathlib import Path
from urllib.robotparser import RobotFileParser
import xml.etree.ElementTree as ET

import pandas as pd
import requests

## 1. Reglas del juego: robots.txt

Antes de pedir nada, revisamos qué permite cada diario. Ambos publican feeds RSS y los listan como canales oficiales.

In [2]:
feeds_larepublica = {
    "home": "https://larepublica.pe/rss/home.xml",
    "politica": "https://larepublica.pe/rss/politica.xml",
    "economia": "https://larepublica.pe/rss/economia.xml",
    "deportes": "https://larepublica.pe/rss/deportes.xml",
    "ciencia": "https://larepublica.pe/rss/ciencia.xml",
}

plantilla_elcomercio = "https://elcomercio.pe/arc/outboundfeeds/google-news-feed/category/{}/?outputType=xml"
feeds_elcomercio = {
    "politica": plantilla_elcomercio.format("politica"),
    "economia": plantilla_elcomercio.format("economia"),
    "deporte-total": plantilla_elcomercio.format("deporte-total"),
    "tecnologia": plantilla_elcomercio.format("tecnologia"),
}

In [3]:
cabeceras = {
    "User-Agent": "TareaDiplomadoPUCP/1.0 (uso academico; contacto: huarhuarojas@gmail.com)"
}
segundos_de_pausa = 1.5

In [4]:
# Pedimos permiso a robots.txt para cada feed que vamos a usar.
# Si algún feed saliera como False, no se descarga.
permisos = []

for medio, sitio, feeds in [
    ("La Republica", "https://larepublica.pe", feeds_larepublica),
    ("El Comercio", "https://elcomercio.pe", feeds_elcomercio),
]:
    lector_robots = RobotFileParser()
    lector_robots.set_url(sitio + "/robots.txt")
    lector_robots.read()
    for nombre, url in feeds.items():
        permitido = lector_robots.can_fetch(cabeceras["User-Agent"], url)
        permisos.append({"medio": medio, "feed": nombre, "permitido": permitido})

pd.DataFrame(permisos)

,medio,feed,permitido
0,La Republica,home,True
1,La Republica,politica,True
2,La Republica,economia,True
3,La Republica,deportes,True
4,La Republica,ciencia,True
5,El Comercio,politica,True
6,El Comercio,economia,True
7,El Comercio,deporte-total,True
8,El Comercio,tecnologia,True


## 2. Leer un feed

Un feed RSS es un XML con una lista de `<item>`. Primero lo vemos con **un** feed.

In [5]:
respuesta = requests.get(feeds_larepublica["politica"], headers=cabeceras, timeout=30)
respuesta.status_code

200

In [6]:
raiz = ET.fromstring(respuesta.content.strip())
notas = raiz.findall("./channel/item")
len(notas)

35

In [7]:
primera_nota = notas[0]

# findtext devuelve el texto de una etiqueta; strip() quita espacios y saltos de línea sobrantes
print("titular:", primera_nota.findtext("title").strip())
print("bajada: ", primera_nota.findtext("description").strip())
print("url:    ", primera_nota.findtext("link").strip())
print("fecha:  ", primera_nota.findtext("pubDate").strip())

titular: Papa León XIV Últimas Noticias: Sumo pontífice ingresó al hospital Gemelli para ser operado de un nódulo pulmonar
bajada:  El pontífice, de 71 años, ingresó al hospital Gemelli de Roma para someterse a la intervención, luego de que los especialistas recomendaran retirarle la lesión, la cual fue identificada en una fase muy temprana y bien localizada.
url:     https://larepublica.pe/politica/2026/10/13/papa-leon-xiv-ultimas-noticias-sumo-pontifice-es-hospitalizado-hoy-para-operacion-de-nodulo-pulmonar-hnews-397824
fecha:   Tue, 13 Oct 2026 14:47:22 GMT


## 3. Repetir para los nueve feeds

Esto se repite nueve veces, así que ahora sí conviene una función.
Regla del enunciado: pausa de 1 a 2 segundos y **máximo un reintento**.

In [8]:
def descargar_feed(medio, url):
    respuesta = None
    for intento in range(2):
        try:
            respuesta = requests.get(url, headers=cabeceras, timeout=30)
            if respuesta.status_code == 200:
                break
        except requests.RequestException:
            respuesta = None
        time.sleep(segundos_de_pausa)

    if respuesta is None or respuesta.status_code != 200:
        print("No se pudo descargar:", url)
        return []

    raiz = ET.fromstring(respuesta.content.strip())
    filas = []
    for nota in raiz.findall("./channel/item"):
        filas.append({
            "titular": (nota.findtext("title") or "").strip(),
            "bajada": (nota.findtext("description") or "").strip(),
            "medio": medio,
            "url": (nota.findtext("link") or "").strip(),
            "fecha": (nota.findtext("pubDate") or "").strip(),
        })
    return filas

In [9]:
todas_las_notas = []

for medio, feeds in [("La Republica", feeds_larepublica), ("El Comercio", feeds_elcomercio)]:
    for nombre, url in feeds.items():
        notas_del_feed = descargar_feed(medio, url)
        todas_las_notas = todas_las_notas + notas_del_feed
        print(medio, nombre, len(notas_del_feed))
        time.sleep(segundos_de_pausa)

crudo = pd.DataFrame(todas_las_notas)
crudo.shape

La Republica home 200


La Republica politica 35


La Republica economia 13


La Republica deportes 17


La Republica ciencia 19


El Comercio politica 42


El Comercio economia 28


El Comercio deporte-total 78


El Comercio tecnologia 20


(452, 5)

## 4. Sección original y categoría estandarizada

La sección es el primer tramo de la URL (`/politica/2026/...`). El mapeo a las cuatro categorías oficiales es explícito.

In [10]:
# split("/")[3] toma el primer tramo después del dominio: https://dominio/SECCION/...
crudo["seccion_original"] = crudo["url"].str.split("/").str[3]
crudo["seccion_original"].value_counts()

seccion_original
politica              94
deporte-total         76
sociedad              60
economia              52
ciencia               30
deportes              27
espectaculos          20
tecnologia            18
mundo                 15
latinos-en-usa        14
opinion                6
verificador            6
cine-series            6
entretenimiento        5
ecdata                 4
publirreportajes       3
cultural               3
carlincatura           2
nota-de-prensa         2
elecciones             2
mastergrama            2
cultura-asiatica       1
horoscopo              1
archivo-elcomercio     1
bbc-news-mundo         1
eldominical            1
Name: count, dtype: int64

In [11]:
mapa_de_categorias = {
    "politica": "politica",
    "economia": "economia",
    "deportes": "deportes",
    "deporte-total": "deportes",
    "ciencia": "ciencia_tecnologia",
    "tecnologia": "ciencia_tecnologia",
}

crudo["etiqueta"] = crudo["seccion_original"].map(mapa_de_categorias)

## 5. Lo que no encaja se descarta y se reporta

Sociedad, espectáculos, mundo, etc. no son una de las cuatro categorías. No inventamos una quinta.

In [12]:
descartadas = crudo[crudo["etiqueta"].isna()]
print("Descartadas:", len(descartadas), "de", len(crudo))
descartadas.groupby(["medio", "seccion_original"]).size().sort_values(ascending=False)

Descartadas: 155 de 452


medio         seccion_original  
La Republica  sociedad              60
              espectaculos          20
              mundo                 15
              latinos-en-usa        14
              verificador            6
              cine-series            6
              entretenimiento        5
El Comercio   ecdata                 4
La Republica  opinion                4
              cultural               3
              publirreportajes       3
              carlincatura           2
El Comercio   elecciones             2
La Republica  nota-de-prensa         2
              mastergrama            2
El Comercio   opinion                2
              archivo-elcomercio     1
              eldominical            1
              bbc-news-mundo         1
La Republica  horoscopo              1
              cultura-asiatica       1
dtype: int64

In [13]:
notas_utiles = crudo[crudo["etiqueta"].notna()].copy()
notas_utiles.shape

(297, 7)

## 6. Limpieza

El mismo artículo puede salir en dos feeds (por ejemplo, `home` y `politica`). La URL identifica la nota.

In [14]:
# Un diario escribe la zona como "GMT" y el otro como "+0000"; format="mixed" acepta ambos.
# Pasamos a hora de Lima y a AAAA-MM-DD, para que un día sea un día.
fecha_utc = pd.to_datetime(notas_utiles["fecha"], utc=True, format="mixed")
notas_utiles["fecha"] = fecha_utc.dt.tz_convert("America/Lima").dt.strftime("%Y-%m-%d")

In [15]:
notas_utiles = notas_utiles[notas_utiles["titular"] != ""]
notas_utiles = notas_utiles.drop_duplicates(subset="url")
notas_utiles.shape

(240, 7)

In [16]:
columnas = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
notas_de_hoy = notas_utiles[columnas]

## 7. Acumular con lo ya guardado

Cada día entran notas nuevas. Juntamos lo de hoy con el CSV anterior y quitamos repetidas por URL.

In [17]:
ruta_csv = Path("datos/noticias.csv")

if ruta_csv.exists():
    acumulado = pd.read_csv(ruta_csv)
else:
    acumulado = pd.DataFrame(columns=columnas)

antes = len(acumulado)
acumulado = pd.concat([acumulado, notas_de_hoy], ignore_index=True)
acumulado = acumulado.drop_duplicates(subset="url", keep="first")

print("Antes:", antes, "| Después:", len(acumulado), "| Nuevas hoy:", len(acumulado) - antes)

Antes: 0 | Después: 240 | Nuevas hoy: 240


In [18]:
# Sin índice y en UTF-8, para que las tildes sobrevivan
ruta_csv.parent.mkdir(exist_ok=True)
acumulado.to_csv(ruta_csv, index=False, encoding="utf-8")

## 8. ¿Cuánto falta? Revisión del dataset

In [19]:
pd.crosstab(acumulado["medio"], acumulado["etiqueta"], margins=True)

etiqueta,ciencia_tecnologia,deportes,economia,politica,All
medio,,,,,
El Comercio,18,76,27,36,157
La Republica,19,17,13,34,83
All,37,93,40,70,240


In [20]:
meta_por_medio = 200
total_por_medio = acumulado["medio"].value_counts()
faltan = meta_por_medio - total_por_medio
faltan.clip(lower=0)

medio
El Comercio      43
La Republica    117
Name: count, dtype: int64

In [21]:
# ¿Hay valores vacíos? La bajada puede faltar en algunas notas.
acumulado.isna().sum()

titular             0
bajada              0
medio               0
seccion_original    0
url                 0
fecha               0
etiqueta            0
dtype: int64

In [22]:
# Rango de fechas cubierto
acumulado.groupby("medio")["fecha"].agg(["min", "max"])

,min,max
medio,,
El Comercio,2026-10-11,2026-10-13
La Republica,2026-10-11,2026-10-13


## Qué hicimos

| Paso | Código |
|---|---|
| Revisar permisos | `RobotFileParser`, `can_fetch` |
| Identificarnos | `headers={"User-Agent": ...}` |
| Leer un feed RSS | `requests.get` + `ET.fromstring` + `findall("./channel/item")` |
| Repetir con pausa y un reintento | `descargar_feed`, `time.sleep` |
| Sección desde la URL | `str.split("/").str[3]` |
| Mapear a 4 categorías | `.map(mapa_de_categorias)` |
| Descartar lo que no encaja | `etiqueta.isna()` |
| Acumular sin repetir | `pd.concat` + `drop_duplicates(subset="url")` |
| Guardar | `to_csv(index=False)` |